In [1]:
from bs4 import BeautifulSoup
import json
import pandas as pd
import os

In [37]:
# Load the HTML content from a file
# file_path = "./HTML_ZZ_name/2xingtangniaobing.html"  # Replace with your actual file path
file_path = "./HTML_ZZ_name/putaotangnailiangjiangdi.html"  # Replace with your actual file path
with open(file_path, 'r', encoding='utf-8') as file:
    html_content = file.read()

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

In [43]:
uls = soup.find_all("li")
for one in uls:
    print(one.get_text(strip=True))

寿险
意外险种
重疾和癌症
失能收入
豁免保费和永久完全性失能
住院收入保障
长期护理险


In [135]:
# Process all tables in the document
tables = soup.find_all("table",class_="table")

In [53]:
# for table in tables[:4]:
#     print(table)
#     print("==========")
#     break
    

In [51]:
home_path = "./HTML_ZZ_name"
for file in os.listdir("./HTML_ZZ_name"):
    print(file)
    break

putaotangnailiangjiangdi.html


In [9]:
def sace_xlsx(save_file,trees,table_names):

    out_save_path = os.path.join("./save/",save_file,save_file+".xlsx")
    with pd.ExcelWriter(out_save_path,engine="openpyxl") as writer:

        for idx, one_tree in enumerate(trees):
            if len(one_tree)==0:
                continue
            # print(one_tree)
            # print(len(one_tree))
            max_level_num = max([len(one_["questions"]) for one_ in one_tree])
            # print(max_level_num)
            data_all = []
            for one_chain in one_tree:
                save_dict = {}
                questions = one_chain["questions"]
                conclustions = one_chain["conclustions"][0]
                for i in range(max_level_num):
                    save_dict[f"{i+1}_级问题"] = questions[i] if i<len(questions) else ''
                for one_dict in conclustions:
                    save_dict[list(one_dict.keys())[0]] = list(one_dict.values())[0]
                # print(one_chain["questions"])
                # print(one_chain["conclustions"][0])
                data_all.append(save_dict)
            df_one = pd.DataFrame(data_all)
            sheetName = table_names[idx]
            # print(f"sheet name len:{len(sheetName)}")
            # print(sheetName)
            df_one.to_excel(writer,sheet_name=sheetName,index=False)


In [10]:
from bs4 import BeautifulSoup
from collections import defaultdict
import json
import os
from tqdm import tqdm



# 用于提取 `padding-left` 值的函数
def extract_padding_left(style):
    """从 style 属性提取 padding-left 的值"""
    if style:
        try:
            for item in style.split(";"):
                if "padding-left" in item:
                    return int(item.split(":")[1].strip().replace("pt", ""))
        except Exception:
            return 0
    return 0

home_path = "./HTML_ZZ_name"
for file in tqdm(os.listdir("./HTML_ZZ_name")):
    file_path = os.path.join(home_path,file)

    save_file = file.strip(".html")
    # print(save_file)
    save_path_file = os.path.join("./save",save_file)
    if not os.path.exists(save_path_file):
            os.mkdir(save_path_file)
    # 加载 HTML 文件
    # print(file_path)
    with open(file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")
    
    try:
        tags = []
        uls = soup.find_all("li")
        for one in uls:
            tags.append(one.get_text(strip=True))
        # 遍历表格
        tables = soup.find_all("table",class_="table")  # 替换为实际表格 ID
        trees = []
        table_names = []
        for index_,table in enumerate(tables):
            # 初始化树结构
            tree = []
            node_stack = {"questions":[],
                        "level":[],
                        "conclustions":[]}
            conc_names = []
            for one_ in table.find_all("tr")[0].find_all("th")[1:]:
                conc_names.append(one_.get_text(strip=True))
            for idx, tr in enumerate(table.find_all("tr")[1:]):  # 跳过表头
                cols = tr.find_all("td")
                    # 提取字段
                if len(cols)>=2:
                    values = []
                    for cols_ in cols[1:]:
                        value = cols_.get_text(strip=True)
                        if value=="":
                            continue
                        values.append(value)
                    div = cols[0].find("div", class_="p")
                    label = div.get_text(strip=True) if div else None
                    style = cols[0].find("div", style=lambda x: x and "padding-left" in x).get("style", "")
                    padding_left = extract_padding_left(style)
                    precedingrows = cols[0].find("div", class_="precedingrows")
                    precedingrows_text = precedingrows.get_text(strip=True) if precedingrows else f"{idx+1}"
                    
                    # 创建当前节点
                    node = {
                        "node_id":idx+1,
                        "label": label,
                        "value": value,
                        "children": [],
                        "padding_left":padding_left,
                        "precedingrows":precedingrows_text
                    }
                    if len(values)==0:
                        # print(padding_left)
                        # print(node_stack)
                        if  len(node_stack["level"])>0:
                            if padding_left>node_stack["level"][-1]:
                                node_stack["level"].append(padding_left)
                                node_stack["questions"].append(label)
                            else:
                                # node_stack["questions"].pop()
                                # node_stack["level"].pop()
                                # print(padding_left)
                                # print(node_stack)
                                while padding_left<=node_stack["level"][-1]:
                                    # print(padding_left)
                                    # print(node_stack)
                                    node_stack["questions"].pop()
                                    node_stack["level"].pop()
                                    if len(node_stack["level"])==0:
                                        break
                                    
                                node_stack["level"].append(padding_left)
                                node_stack["questions"].append(label)
                        else:
                            node_stack["level"].append(padding_left)
                            node_stack["questions"].append(label)
                    if len(values)!=0:
                        # print(padding_left)
                        # print(node_stack)
                        if len(node_stack["level"])>0:
                            while padding_left<=node_stack["level"][-1]:
                                node_stack["questions"].pop()
                                node_stack["level"].pop()
                                if len(node_stack["level"])==0:
                                        break
                        # print(padding_left)
                        # print(node_stack)
                        node_stack["level"].append(padding_left)
                        node_stack["questions"].append(label)
                        conc_save = [{conc_names[i]:values[i]} for i in range(len(values))]
                        node_stack["conclustions"].append(conc_save)
                        # print(node_stack)
                        tree.append(node_stack)
                        # node_stack["questions"].pop()
                        node_stack = {"questions":node_stack["questions"].copy(),
                                    "level":node_stack["level"].copy(),
                        "conclustions":[]}
                
            
            
            # to_save_file_path = os.path.join(save_path_file,"#".join(conc_names)+".json")
            to_save_file_path = os.path.join(save_path_file,tags[index_]+".json") 
            trees.append(tree)
            # print(conc_names)
            table_names.append(tags[index_])
            with open(to_save_file_path,"w")as f:
                json.dump(tree,f,ensure_ascii=False,indent=2)
        sace_xlsx(save_file,trees,tags)    
    except Exception as es:
        print("read error :",file_path)
        print(es)        
                

# 打印结果
# print(json.dumps(tree, ensure_ascii=False, indent=2))


100%|██████████| 376/376 [00:58<00:00,  6.43it/s]


In [5]:
def soup_html(file_path):
    # 加载 HTML 文件
    print(file_path)
    with open(file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")
    
    try:
        tags = []
        uls = soup.find_all("li")
        for one in uls:
            tags.append(one.get_text(strip=True))
        # 遍历表格
        tables = soup.find_all("table",class_="table")  # 替换为实际表格 ID
        trees = []
        table_names = []
        for index_,table in enumerate(tables):
            # 初始化树结构
            tree = []
            node_stack = {"questions":[],
                        "level":[],
                        "conclustions":[]}
            conc_names = []
            for one_ in table.find_all("tr")[0].find_all("th")[1:]:
                conc_names.append(one_.get_text(strip=True))
            for idx, tr in enumerate(table.find_all("tr")[1:]):  # 跳过表头
                cols = tr.find_all("td")
                    # 提取字段
                if len(cols)>=2:
                    values = []
                    for cols_ in cols[1:]:
                        value = cols_.get_text(strip=True)
                        if value=="":
                            continue
                        values.append(value)
                    div = cols[0].find("div", class_="p")
                    label = div.get_text(strip=True) if div else None
                    style = cols[0].find("div", style=lambda x: x and "padding-left" in x).get("style", "")
                    padding_left = extract_padding_left(style)
                    precedingrows = cols[0].find("div", class_="precedingrows")
                    precedingrows_text = precedingrows.get_text(strip=True) if precedingrows else f"{idx+1}"
                    
                    # 创建当前节点
                    node = {
                        "node_id":idx+1,
                        "label": label,
                        "value": value,
                        "children": [],
                        "padding_left":padding_left,
                        "precedingrows":precedingrows_text
                    }
                    if len(values)==0:
                        # print(padding_left)
                        # print(node_stack)
                        if  len(node_stack["level"])>0:
                            if padding_left>node_stack["level"][-1]:
                                node_stack["level"].append(padding_left)
                                node_stack["questions"].append(label)
                            else:
                                # node_stack["questions"].pop()
                                # node_stack["level"].pop()
                                # print(padding_left)
                                # print(node_stack)
                                while padding_left<=node_stack["level"][-1]:
                                    # print(padding_left)
                                    # print(node_stack)
                                    node_stack["questions"].pop()
                                    node_stack["level"].pop()
                                    if len(node_stack["level"])==0:
                                        break
                                    
                                node_stack["level"].append(padding_left)
                                node_stack["questions"].append(label)
                        else:
                            node_stack["level"].append(padding_left)
                            node_stack["questions"].append(label)
                    if len(values)!=0:
                        # print(padding_left)
                        # print(node_stack)
                        if len(node_stack["level"])>0:
                            while padding_left<=node_stack["level"][-1]:
                                node_stack["questions"].pop()
                                node_stack["level"].pop()
                                if len(node_stack["level"])==0:
                                        break
                        # print(padding_left)
                        # print(node_stack)
                        node_stack["level"].append(padding_left)
                        node_stack["questions"].append(label)
                        conc_save = [{conc_names[i]:values[i]} for i in range(len(values))]
                        node_stack["conclustions"].append(conc_save)
                        # print(node_stack)
                        tree.append(node_stack)
                        # node_stack["questions"].pop()
                        node_stack = {"questions":node_stack["questions"].copy(),
                                    "level":node_stack["level"].copy(),
                        "conclustions":[]}
                
            
            
            # to_save_file_path = os.path.join(save_path_file,"#".join(conc_names)+".json")
            to_save_file_path = os.path.join(save_path_file,tags[index_]+".json") 
            trees.append(tree)
            # print(conc_names)
            table_names.append(tags[index_])
            with open(to_save_file_path,"w")as f:
                json.dump(tree,f,ensure_ascii=False,indent=2)
        return trees,tags
    except Exception as es:
        print("read error :",file_path)
        print(es)
        return [],[]

In [6]:
file_path = "./HTML_ZZ_name/feiyizhi.html"
soup_html(file_path)

./HTML_ZZ_name/feiyizhi.html


([[{'questions': ['所有案例'],
    'level': [0],
    'conclustions': [[{'寿险': '咨询首席核保师；通常拒保。对某些特殊案例（需要综合考虑病因、病程、预后等情况）可以考虑评点，最低评点+400点'}]]}],
  [{'questions': ['所有案例'],
    'level': [0],
    'conclustions': [[{'意外死亡': '拒保'}, {'意外残疾': '拒保'}]]}],
  [{'questions': ['所有案例'],
    'level': [0],
    'conclustions': [[{'重疾': '拒保'}, {'癌症': '拒保'}]]}],
  [{'questions': ['所有案例'],
    'level': [0],
    'conclustions': [[{'失能收入保障-4周等待期': '拒保'}, {'失能收入保障-13周等待期': '拒保'}]]}],
  [{'questions': ['所有案例'],
    'level': [0],
    'conclustions': [[{'豁免保费': '拒保'},
      {'永久完全性失能-职业': '拒保'},
      {'永久完全性失能-基本日常生活活动': '拒保'}]]}],
  [],
  [{'questions': ['所有案例'],
    'level': [0],
    'conclustions': [[{'长期护理险': '拒保'}]]}]],
 ['寿险', '意外险种', '重疾和癌症', '失能收入', '豁免保费和永久完全性失能', '住院收入保障', '长期护理险'])

In [34]:
print(len(trees))
print(table_names)

7
['寿险', '意外死亡#意外残疾', '重疾#癌症', '失能收入保障-4周等待期#失能收入保障-13周等待期', '豁免保费#永久完全性失能-职业#永久完全性失能-基本日常生活活动', '住院收入保障', '长期护理险']


In [27]:
print(save_file)

putaotangnailiangjiangdi


In [47]:

out_save_path = os.path.join("./save/",save_file,save_file+".xlsx")
with pd.ExcelWriter(out_save_path,engine="openpyxl") as writer:

    for idx, one_tree in enumerate(trees):
        # print(one_tree)
        # print(len(one_tree))
        max_level_num = max([len(one_["questions"]) for one_ in one_tree])
        print(max_level_num)
        data_all = []
        for one_chain in one_tree:
            save_dict = {}
            questions = one_chain["questions"]
            conclustions = one_chain["conclustions"][0]
            for i in range(max_level_num):
                save_dict[f"{i+1}_级问题"] = questions[i] if i<len(questions) else ''
            for one_dict in conclustions:
                save_dict[list(one_dict.keys())[0]] = list(one_dict.values())[0]
            # print(one_chain["questions"])
            # print(one_chain["conclustions"][0])
            data_all.append(save_dict)
        df_one = pd.DataFrame(data_all)
        sheetName = table_names[idx]
        print(f"sheet name len:{len(sheetName)}")
        print(sheetName)
        df_one.to_excel(writer,sheet_name=sheetName,index=False)


3
sheet name len:2
寿险
3
sheet name len:4
意外险种
3
sheet name len:5
重疾和癌症
3
sheet name len:4
失能收入
3
sheet name len:12
豁免保费和永久完全性失能
3
sheet name len:6
住院收入保障
3
sheet name len:5
长期护理险


In [26]:
print(data_all)

[{'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白异常', '3_级问题': '', '豁免保费': '按2型糖尿病进行评点', '永久完全性失能-职业': '按2型糖尿病进行评点', '永久完全性失能-基本日常生活活动': '按2型糖尿病进行评点'}, {'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白正常', '3_级问题': '≤ 39岁', '豁免保费': '拒保', '永久完全性失能-职业': '拒保', '永久完全性失能-基本日常生活活动': '拒保'}, {'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白正常', '3_级问题': '40 – 49岁', '豁免保费': '+100', '永久完全性失能-职业': '+100', '永久完全性失能-基本日常生活活动': '+75'}, {'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白正常', '3_级问题': '50 – 59岁', '豁免保费': '+50', '永久完全性失能-职业': '+50', '永久完全性失能-基本日常生活活动': '+25'}, {'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白正常', '3_级问题': '≥ 60岁', '豁免保费': '+25', '永久完全性失能-职业': '+25', '永久完全性失能-基本日常生活活动': '+25'}, {'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白值不详', '3_级问题': '≤ 39岁', '豁免保费': '拒保', '永久完全性失能-职业': '拒保', '永久完全性失能-基本日常生活活动': '拒保'}, {'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白值不详', '3_级问题': '40 – 49岁', '豁免保费': '+125', '永久完全性失能-职业': '+125', '永久完全性失能-基本日常生活活动': '+100'}, {'1_级问题': '葡萄糖耐量降低', '2_级问题': '糖化血红蛋白值不详', '3_级问题': '50 – 59岁', '豁免保费': '+75', '永久完全性失能-职业': '+75', '永久完全

In [30]:
!pip install openpyxl

In [11]:
output = "{\"viscera\": {\"内部回声\": \"均匀\", \"包膜\": \"光整\", \"脏器大小\": \"正常\", \"峡部大小\": \"\", \"形态\": \"\", \"表面\": \"\"}, \"occ_lesion\": [{\"类型\": \"结节\", \"TI-RADS分级\": \"\", \"位置\": \"甲状腺左侧叶\", \"别名\": \"\", \"回声强弱\": \"低回声\", \"大小\": \"结节不超过1cm\", \"性状特征\": \"\", \"数目\": \"单个\", \"轮廓\": \"\", \"边界\": \"尚清\", \"形态\": \"欠规则\", \"横断面状态\": \"\"}], \"Conclusion\": [\"甲状结节\"], \"CDFI\": \"未见血流信号\"}"
output_dict = eval(output)

In [12]:
output_dict

{'viscera': {'内部回声': '均匀',
  '包膜': '光整',
  '脏器大小': '正常',
  '峡部大小': '',
  '形态': '',
  '表面': ''},
 'occ_lesion': [{'类型': '结节',
   'TI-RADS分级': '',
   '位置': '甲状腺左侧叶',
   '别名': '',
   '回声强弱': '低回声',
   '大小': '结节不超过1cm',
   '性状特征': '',
   '数目': '单个',
   '轮廓': '',
   '边界': '尚清',
   '形态': '欠规则',
   '横断面状态': ''}],
 'Conclusion': ['甲状结节'],
 'CDFI': '未见血流信号'}